# 03 - ABMIL baseline training

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/koushikkkkkkkkkk/prostate-mil-project/blob/main/notebooks/03_abmil_training.ipynb)

**Owner:** Person 1 (Team Leader & ABMIL Baseline Owner)

Train the gated-attention ABMIL baseline that CLAM-SB is compared against. Same
split, same features, same seeds as every other model -- that is what makes the
comparison paired and therefore statistically meaningful.


## Setup

In [ ]:
# If running in Google Colab, clone repo and install dependencies:
try:
    import google.colab
    IN_COLAB = True
    import os
    if not os.path.exists("prostate-mil-project"):
        !git clone https://github.com/koushikkkkkkkkkk/prostate-mil-project.git
    %cd prostate-mil-project
    !git pull origin main
    !pip install -r requirements.txt
except ImportError:
    IN_COLAB = False
print("Running in Colab:", IN_COLAB)


In [ ]:
import os, sys, json
from pathlib import Path
os.environ.setdefault("MPLBACKEND", "Agg")
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
import numpy as np, pandas as pd, torch
print("torch", torch.__version__)

In [ ]:
MANIFEST = ROOT / "data/mock/manifest.csv" if (ROOT / "data/mock/manifest.csv").exists() else ROOT / "data/common_split.csv"
FEATURES = MANIFEST.parent / "features"
CONFIG = ROOT / ("configs/abmil_mock.yaml" if "mock" in str(MANIFEST) else "configs/abmil.yaml")
OUT = ROOT / ("results/mock_abmil" if "mock" in str(MANIFEST) else "results/metrics/abmil")
SYNTHETIC = "mock" in str(MANIFEST)
print(MANIFEST, CONFIG, OUT, "synthetic:", SYNTHETIC)

## Train (3 seeds on real data, 1 seed on mock)

In [ ]:
import subprocess, sys
seeds = ["0"] if SYNTHETIC else ["0", "1", "2"]
cmd = [sys.executable, "-m", "src.train", "--model", "abmil",
       "--manifest", str(MANIFEST), "--features-root", str(FEATURES),
       "--config", str(CONFIG), "--out-dir", str(OUT), "--seeds", *seeds]
print(" ".join(cmd))
res = subprocess.run(cmd, cwd=ROOT, capture_output=True, text=True)
print(res.stdout[-2000:]); print(res.stderr[-800:]); print(res.returncode)

## Results and confusion matrix

In [ ]:
aggregate = json.loads((OUT / "aggregate_abmil.json").read_text())
print((OUT / "table_abmil.md").read_text())
pd.DataFrame(aggregate["per_run"])

In [ ]:
from IPython.display import Image, display
pngs = sorted(OUT.glob("confusion_*.png"))
display(Image(filename=str(pngs[0]), width=420)) if pngs else print("no confusion matrix yet")

## What Person 2 needs from this notebook

* `results/metrics/abmil/predictions_abmil_seed0.csv` -- the prediction file used
  in the paired CLAM-SB vs ABMIL comparison.
* The aggregate QWK, so the comparison can quote both numbers side by side.